# Local normalization programmes for Barnette, Tutte 5-flow, and Berge–Fulkerson

This notebook records theorem-level deductions and reproducible finite tests
obtained by applying a Collatz-style rewriting viewpoint to three open graph
conjectures.

**Status:** none of the three conjectures is solved here. The strongest outputs are:

1. an exact topological defect identity for Barnette colourings;
2. an explicit counterexample to the one/two-face normalization rule and a
   surviving three-face normalization conjecture;
3. a residual-cycle optimality and polarized-cut theorem for integer 6-flows;
4. unconditional and conditional quadratic descent theorems for six perfect
   matchings;
5. extensive exhaustive evidence for one-matching normalization in
   Berge–Fulkerson state spaces.

Large computations are recorded in the accompanying source-and-log archive.
The executable cells below verify the local identities, the explicit Barnette
trap and escape, the Tutte algebra, and the Petersen normalization test.

In [1]:
import sys
import subprocess
import importlib.util

for module, package in {
    "networkx": "networkx>=3.2",
    "sympy": "sympy>=1.12",
    "pandas": "pandas>=2.0",
}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from collections import Counter, defaultdict, deque
from itertools import combinations_with_replacement
import math
import networkx as nx
import pandas as pd
import sympy as sp

print("NetworkX:", nx.__version__)

NetworkX: 3.6.1


# Part I — Barnette

Let \(G\) be a cubic plane bipartite graph and let \(H=G^*\) be its
Eulerian triangulation dual. For a nontrivial binary colouring

\[
V(H)=A\sqcup B,
\]

put

\[
\psi(F)=c(F)-1+\beta(F),
\qquad
\beta(F)=|E(F)|-|V(F)|+c(F),
\]

and

\[
\Phi(A,B)=\psi(H[A])+\psi(H[B]).
\]

Let \(X\subseteq E(G)\) be the primal edges dual to the bichromatic edges
\(\delta_H(A)\). Every triangular face of \(H\) contains zero or two
bichromatic edges, so \(X\) is an even subgraph of the cubic graph \(G\):
a vertex-disjoint union of cycles.

Write:

* \(z(X)\) for the number of vertices of \(G\) omitted by \(X\);
* \(t(X)\) for the number of cycle components of \(X\).

## Exact defect identity

\[
\boxed{\Phi(A,B)=z(X)+2(t(X)-1).}
\]

### Proof

If \(h=|V(H)|\), then

\[
|E(H)|=3h-6,\qquad |V(G)|=2h-4.
\]

Since \(X\) is 2-regular on \(2h-4-z\) vertices,

\[
|\delta_H(A)|=|E(X)|=2h-4-z.
\]

Hence the total number of monochromatic edges is

\[
|E(H[A])|+|E(H[B])|
=(3h-6)-(2h-4-z)=h-2+z.
\]

The \(t\) disjoint Jordan curves of \(X\) divide the sphere into \(t+1\)
regions. Those regions are exactly the connected monochromatic pieces, so

\[
c(H[A])+c(H[B])=t+1.
\]

Substitution into the definition of \(\Phi\) gives the formula.

Thus \(\Phi=0\) exactly when \(X\) is one spanning cycle, i.e. a
Hamiltonian cycle of \(G\).

In [2]:
def plane_faces(G):
    planar, embedding = nx.check_planarity(G)
    assert planar
    seen = set()
    faces = []
    for u, v in embedding.edges():
        if (u, v) not in seen:
            faces.append(embedding.traverse_face(u, v, seen))
    return faces, embedding


def dual_with_edge_map(G):
    faces, _ = plane_faces(G)
    edge_faces = defaultdict(list)
    for i, face in enumerate(faces):
        for j, u in enumerate(face):
            v = face[(j + 1) % len(face)]
            edge_faces[tuple(sorted((u, v)))].append(i)

    H = nx.Graph()
    H.add_nodes_from(range(len(faces)))
    dual_to_primal = {}
    for primal_edge, indices in edge_faces.items():
        assert len(indices) == 2
        a, b = indices
        H.add_edge(a, b)
        dual_to_primal[tuple(sorted((a, b)))] = primal_edge
    return H, dual_to_primal


def side_psi(H, vertices):
    vertices = set(vertices)
    if not vertices:
        raise ValueError("Use only nontrivial colourings.")
    F = H.subgraph(vertices)
    c = nx.number_connected_components(F)
    beta = F.number_of_edges() - F.number_of_nodes() + c
    return (c - 1) + beta


def barnette_quantities(G, mask):
    H, dual_to_primal = dual_with_edge_map(G)
    n = H.number_of_nodes()
    A = {v for v in H if (mask >> v) & 1}
    B = set(H) - A
    if not A or not B:
        raise ValueError("Trivial colouring.")

    phi = side_psi(H, A) + side_psi(H, B)
    X = []
    for u, v in H.edges():
        if ((mask >> u) & 1) != ((mask >> v) & 1):
            X.append(dual_to_primal[tuple(sorted((u, v)))])

    F = nx.Graph()
    F.add_nodes_from(G)
    F.add_edges_from(X)
    nonisolated = [v for v, d in F.degree() if d > 0]
    z = G.number_of_nodes() - len(nonisolated)
    t = nx.number_connected_components(F.subgraph(nonisolated)) if nonisolated else 0
    return phi, z, t


# Verify the identity for every nontrivial colouring of the cube dual.
cube = nx.cubical_graph()
H_cube, _ = dual_with_edge_map(cube)
for mask in range(1, (1 << H_cube.number_of_nodes()) - 1):
    phi, z, t = barnette_quantities(cube, mask)
    assert phi == z + 2 * (t - 1)

print(
    "Exact Barnette identity verified on all",
    (1 << H_cube.number_of_nodes()) - 2,
    "nontrivial cube-dual colourings."
)

Exact Barnette identity verified on all 62 nontrivial cube-dual colourings.


## The failed two-face rule and a three-face escape

A natural rewrite flips a connected patch of at most \(k\) dual vertices,
allowing the move only when \(\Phi\) does not increase.

The rule with \(k=2\) is false. The following 36-vertex Barnette graph has
six colourings trapped under all non-increasing one- and two-face flips.
One trap is repaired by a connected three-face flip followed by a two-face
flip.

In [3]:
barnette36_graph6 = (
    "c?_OG?@?gG`@_?O?_G??DO??_??_?C??S??G??@H???_O?DO????O??_?"
    "C?G?A?D?@????_??G??_?GO???S??C?????_?G_????c????D"
)
G36 = nx.from_graph6_bytes(barnette36_graph6.encode())
H36, _ = dual_with_edge_map(G36)

assert G36.number_of_nodes() == 36
assert set(dict(G36.degree()).values()) == {3}
assert nx.is_bipartite(G36)
assert nx.node_connectivity(G36) >= 3

def colouring_phi(H, mask):
    A = {v for v in H if (mask >> v) & 1}
    B = set(H) - A
    if not A or not B:
        return H.number_of_nodes() + 1
    return side_psi(H, A) + side_psi(H, B)

path = [176722, 176212, 188500]
patches = [
    {v for v in H36 if ((path[0] ^ path[1]) >> v) & 1},
    {v for v in H36 if ((path[1] ^ path[2]) >> v) & 1},
]

print("Dual order:", H36.number_of_nodes())
print("Potential values:", [colouring_phi(H36, mask) for mask in path])
print("Flipped patches:", patches)
print("Patch connectedness:", [nx.is_connected(H36.subgraph(P)) for P in patches])

assert [colouring_phi(H36, mask) for mask in path] == [2, 2, 0]
assert len(patches[0]) == 3 and nx.is_connected(H36.subgraph(patches[0]))
assert len(patches[1]) == 2 and nx.is_connected(H36.subgraph(patches[1]))

Dual order: 20
Potential values: [2, 2, 0]
Flipped patches: [{1, 2, 9}, {12, 14}]
Patch connectedness: [True, True]


### Barnette Three-Face Normalization Conjecture

For every Eulerian plane triangulation, every nontrivial binary colouring
can be transformed to a two-tree partition by non-increasing flips of
connected patches of at most three vertices.

Equivalently in the primal: every even subgraph of a Barnette graph can
be transformed into a Hamiltonian cycle by symmetric differences with
boundaries of connected patches of at most three faces, never increasing

\[
z+2(t-1).
\]

Recorded exhaustive tests cover 80 generated Barnette graphs through
40 vertices and **139,796,480 complete colouring states**. One/two-face
traps occur, but every tested state normalizes with patches of size at
most three. A separate exhaustive two-face test on a 48-vertex graph found
six traps; explicit local searches opened each with connected three-face
moves. These are computational findings, not a proof.

# Part II — Tutte's 5-flow conjecture

Fix an orientation and an integer nowhere-zero 6-flow

\[
f:E(G)\to\{1,2,3,4,5\}.
\]

Let

\[
\nu(f)=|\{e:f(e)=5\}|.
\]

For an oriented edge \(x\to y\), define residual arcs:

* \(x\to y\) when \(f(e)\le4\), with cost \(1\) if \(f(e)=4\), otherwise \(0\);
* \(y\to x\) when \(f(e)\ge2\), with cost \(-1\) if \(f(e)=5\), otherwise \(0\).

Augmenting one unit around a directed residual cycle preserves all values in
\(\{1,\dots,5\}\), and the change in \(\nu\) equals the residual-cycle cost.

## Residual optimality theorem

The following are equivalent:

1. no residual cycle has negative cost;
2. there is an integer potential \(h:V(G)\to\mathbb Z\) satisfying, on every
   oriented edge \(x\to y\),

\[
\begin{array}{c|c}
f(e)&h(y)-h(x)\\ \hline
1&\le0\\
2,3&=0\\
4&\in\{0,1\}\\
5&\ge1.
\end{array}
\]

This is the standard negative-cycle/potential duality applied to this
special residual network. Because the edge penalty
\(\mathbf 1_{\{f(e)=5\}}\) has nondecreasing discrete marginal costs, a
negative-cycle-free state minimizes \(\nu\) throughout its fixed
orientation/sign cell.

For every threshold set

\[
S_k=\{v:h(v)\ge k\},
\]

the crossing edges are polarized:

* value-1 edges point out of \(S_k\);
* value-4 and value-5 edges point into \(S_k\);
* value-2 and value-3 edges do not cross.

If \(a\) value-4 edges and \(c\) value-5 edges enter, flow conservation
forces \(4a+5c\) value-1 edges to leave. Hence

\[
\boxed{|\delta(S_k)|=5a+6c.}
\]

In the one-defect case \(c=1\), every threshold crossed by the defect has
size \(6+5a\), congruent to \(1\pmod5\).

In [4]:
a, c = sp.symbols("a c", integer=True, nonnegative=True)
b = 4*a + 5*c
cut_size = sp.simplify(a + b + c)
print("|delta(S)| =", cut_size)
assert sp.expand(cut_size - (5*a + 6*c)) == 0

|delta(S)| = 5*a + 6*c


This gives a rigorous rewrite-or-obstruction theorem: a value-5 defect is
removed by a residual cycle, or a nested family of polarized cuts certifies
local optimality. It does not yet exclude such cuts in a minimum counterexample
to Tutte's conjecture.

# Part III — Berge–Fulkerson

A state is a multiset of six perfect matchings

\[
\mathcal M=(M_1,\dots,M_6).
\]

Let \(m(e)\) be the number of these matchings containing \(e\), and define

\[
Q(\mathcal M)=\sum_{e\in E(G)}(m(e)-2)^2.
\]

Then \(Q=0\) exactly for a Fulkerson cover.

## Unconditional one-replacement descent theorem

Let \(G\) be a bridgeless cubic graph on \(n\) vertices. Choose one current
matching uniformly, remove it, and add a random perfect matching whose edge
marginals are \(1/3\). Such a distribution exists because the all-\(1/3\)
vector belongs to the perfect-matching polytope.

A direct edgewise calculation gives

\[
\boxed{\mathbb E[\Delta Q]=\frac{2n-Q}{3}.}
\]

Consequently:

\[
\boxed{Q>2n\Longrightarrow
\text{some single-matching replacement strictly decreases }Q.}
\]

Every one-replacement local minimum therefore lies in the bounded
low-defect region \(Q\le2n\).

In [5]:
d = sp.symbols("d", integer=True)
m = d + 2

# I: removed current matching contains the edge, P(I=1)=m/6.
# J: added random matching contains the edge, P(J=1)=1/3.
# They are independent.
E_Z = -m/sp.Integer(6) + sp.Rational(1, 3)
E_Z2 = m/sp.Integer(6) + sp.Rational(1, 3) - 2*(m/sp.Integer(6))*sp.Rational(1, 3)
edge_delta = sp.expand(2*d*E_Z + E_Z2)

print("Expected edge contribution:", sp.factor(edge_delta))
# Summing uses sum d_e=0 and |E|=3n/2.
Q, n = sp.symbols("Q n", nonnegative=True)
total = sp.simplify(-Q/sp.Integer(3) + sp.Rational(4, 9)*(3*n/sp.Integer(2)))
print("Expected total change:", total)
assert sp.simplify(total - (2*n-Q)/3) == 0

Expected edge contribution: -(6*d**2 - d - 8)/18
Expected total change: -Q/3 + 2*n/3


## Conditional \(k\)-replacement descent

Assume a Fulkerson cover exists. Remove \(k\) current matchings uniformly
without replacement and add \(k\) matchings from a fixed Fulkerson cover,
also uniformly without replacement. Hypergeometric variance gives

\[
\boxed{
\mathbb E[\Delta Q]
=
\frac{k}{30}
\left((k-11)Q+4(6-k)n\right).
}
\]

Therefore a strict \(k\)-replacement exists above the thresholds

\[
\begin{array}{c|ccccc}
k&1&2&3&4&5\\ \hline
Q/n&2&16/9&3/2&8/7&2/3.
\end{array}
\]

The theorem isolates the genuinely hard region of the state space.

In [6]:
k = sp.symbols("k", integer=True, positive=True)
d = sp.symbols("d", integer=True)
m = d + 2

var_removed = k*(6-k)*m*(6-m)/(36*5)
var_added = k*(6-k)*2*4/(36*5)
mean_z = -k*d/6
edge_change = sp.expand(2*d*mean_z + var_removed + var_added + mean_z**2)
print("Edgewise:", sp.factor(edge_change))

formula = k/sp.Integer(30) * ((k-11)*Q + 4*(6-k)*n)
print("Total:", formula)

thresholds = {
    1: sp.Rational(2, 1),
    2: sp.Rational(16, 9),
    3: sp.Rational(3, 2),
    4: sp.Rational(8, 7),
    5: sp.Rational(2, 3),
}
pd.DataFrame(
    [{"k": kk, "strict descent when Q/n >": str(v)}
     for kk, v in thresholds.items()]
)

Edgewise: k*(3*d**2*k - 33*d**2 - d*k + 6*d - 8*k + 48)/90
Total: k*(Q*(k - 11) + n*(24 - 4*k))/30


,k,strict descent when Q/n >
0,1,2
1,2,16/9
2,3,3/2
3,4,8/7
4,5,2/3


## Exhaustive Petersen normalization

The Petersen graph has six perfect matchings. Hence there are

\[
\binom{6+6-1}{6}=462
\]

six-matching multisets. The following cell constructs all states and verifies
that every one reaches a Fulkerson cover through non-increasing replacement
of one matching at a time.

In [7]:
def perfect_matchings(G):
    nodes = tuple(G.nodes())
    result = []

    def rec(remaining, chosen):
        if not remaining:
            result.append(frozenset(tuple(sorted(e)) for e in chosen))
            return
        u = min(remaining)
        for v in G.neighbors(u):
            if v in remaining:
                rec(
                    remaining - {u, v},
                    chosen + [(u, v)],
                )

    rec(set(nodes), [])
    return sorted(set(result), key=lambda M: sorted(M))


def state_Q(edge_list, pms, state):
    mult = Counter()
    for index in state:
        mult.update(pms[index])
    return sum((mult[e] - 2)**2 for e in edge_list)


P = nx.petersen_graph()
edges = sorted(tuple(sorted(e)) for e in P.edges())
pms = perfect_matchings(P)
states = list(combinations_with_replacement(range(len(pms)), 6))
qvals = {state: state_Q(edges, pms, state) for state in states}
targets = [state for state in states if qvals[state] == 0]

reverse = defaultdict(list)
for state in states:
    current = list(state)
    for position in range(6):
        base = current[:position] + current[position+1:]
        for replacement in range(len(pms)):
            new_state = tuple(sorted(base + [replacement]))
            if qvals[new_state] <= qvals[state]:
                reverse[new_state].append(state)

reached = set(targets)
queue = deque(targets)
while queue:
    state = queue.popleft()
    for predecessor in reverse[state]:
        if predecessor not in reached:
            reached.add(predecessor)
            queue.append(predecessor)

print("Perfect matchings:", len(pms))
print("States:", len(states))
print("Fulkerson covers:", len(targets))
print("States reaching a cover:", len(reached))

assert len(pms) == 6
assert len(states) == 462
assert len(reached) == len(states)

Perfect matchings: 6
States: 462
Fulkerson covers: 1
States reaching a cover: 462


## Larger recorded state-space tests

The accompanying archive contains the exact C++ engines and output logs.

* Flower snark \(J_5\): 32 perfect matchings,
  **2,324,784** six-matching multisets; all normalized by non-increasing
  one-matching replacements.
* 29 random bridgeless cubic graphs on 12–16 vertices:
  **945,813** states; no unresolved state.
* 15 random bipartite cubic graphs on 12–14 vertices:
  **6,664,385** states; no unresolved state.
* All connected bridgeless cubic isomorphism classes through ten vertices
  were also tested. One non-snark, 3-edge-colourable graph has a
  one-replacement trap at \(Q=6\); replacing two matchings resolves it.
  No snark obstruction was found.

This motivates the stronger, still open:

### Snark One-Replacement Normalization Conjecture

For every snark, every multiset of six perfect matchings reaches a Fulkerson
cover through replacements of one matching at a time, never increasing \(Q\).

It implies Berge–Fulkerson.

## A polyhedral class theorem

Let \(G\) be a bridgeless cubic matching-covered graph. Suppose every component
in its tight-cut decomposition is either:

* a Birkhoff–von Neumann component, or
* a Petersen brick.

Then \(G\) has a Fulkerson cover.

For a Birkhoff–von Neumann component, the perfect-matching polytope has the
integer decomposition property. Since the all-\(1/3\) vector is a fractional
perfect matching, the integral vector \(2\mathbf 1\in6P\) decomposes into six
perfect matchings. The Petersen graph has its standard six-matchings,
double-covering every edge.

Across a tight cut of a cubic graph, every perfect matching uses exactly one
of the three cut edges. In either six-cover, each cut edge occurs twice.
Relabel the six matchings on the two sides so that equal cut-edge choices are
paired, then splice them. Iterating over the tight-cut decomposition produces
a Fulkerson cover of \(G\).

This is a rigorous class theorem, though it is likely best viewed as a concise
corollary of known integer-decomposition and tight-cut composition facts until
a deeper novelty check says otherwise.

# Comparative research verdict

* **Barnette:** strongest new rewriting conjecture and strongest topological
  state model. The three-face rule has striking evidence, but no proof.
* **Tutte 5-flow:** cleanest rigorous obstruction theorem, but it is close to
  min-cost circulation duality and does not yet eliminate the obstruction cuts.
* **Berge–Fulkerson:** strongest theorem-producing direction. The quadratic
  descent theorem is unconditional, the low-defect region is sharply isolated,
  and one-replacement normalization has survived millions of exact states.
  The next theorem target is to classify \(Q\le2n\) local minima in snarks.